# 03 · "Is this a crop leaf photo?" gate + rare-class correction → ExecuTorch
**Bharat AI-SoC 2026–27 · PS1 · IlaiNalam**

Two jobs, one run (~40 min):
1. **Leaf-photo gate:** a tiny MobileNetV3-Small (≈1 MB INT8) that answers *crop leaf photo / not a leaf photo*. It runs before the disease model, so people, sports fields, rooms and Holi-green faces are rejected **even when they are green**.
   *Yes* = PlantVillage leaves (some pasted on new backgrounds) + PlantDoc field photos. *No* = ~5,000 everyday COCO photos (people, crowds, sports, streets, rooms, animals), faces, textures, plain backgrounds, and **green-tinted / green-paint copies** of them.
2. **Rare-class correction for the v3 disease model (no retraining):** PlantVillage is lopsided (e.g. ~14× more *leaf curl virus* than *mosaic virus* photos), so the model leans to common classes. We measure how much to boost rare classes (`prior_tau`) on validation photos and check the effect on test photos.

**Kaggle setup:**
1. *Create ▸ New Notebook ▸ File ▸ Import Notebook* → this file.
2. *Session options*: **GPU T4 x2**, **Internet → On**.
3. *Add Input*: **"plantvillage dataset"** (abdallahalidev) — as before.
4. *Add Input ▸ Upload*: your **`ps1_vision_outputs_v3.zip`** as a new dataset (any name). The notebook finds `crop_disease_int8.pte` inside it.
5. **Run All**, then download **`ps1_gate_outputs.zip`** (contains `leaf_gate_int8.pte` + `model_config.json` for the app).

**The disease model is NOT changed** — its accuracy numbers stay as they are.

In [ ]:
# Uses Kaggle's pre-installed torch/torchvision. Do NOT pip-install torch or executorch in this kernel.
!pip install -q --default-timeout=120 --retries 10 kagglehub opencv-python-headless scikit-learn
import os, sys, json, random, glob, time, shutil, subprocess, re, io
import numpy as np, torch, torchvision, cv2
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Accelerator to GPU T4 x2")

In [ ]:
# ---- Config -------------------------------------------------------------
SEED, IMG_SIZE, BATCH, EPOCHS, LR = 42, 224, 64, 4, 1e-3
BG_SWAP_PROB, PLANTDOC_REPEAT = 0.5, 2
N_PV_POS = 12000            # PlantVillage training leaves used as "yes" examples
IS_KAGGLE = os.path.exists("/kaggle/working")
WORK      = "/kaggle/working" if IS_KAGGLE else "/content"
OUT_DIR   = os.environ.get("OUT_DIR_OVERRIDE", f"{WORK}/out_gate")
ENV_DIR   = os.environ.get("ET_ENV_DIR", "/tmp/et_env")
TMP       = os.environ.get("TMP_DIR", "/tmp/ps1")
FAST_TEST = os.environ.get("FAST_TEST") == "1"               # local smoke test only
if FAST_TEST: EPOCHS, N_PV_POS = 1, 200
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(TMP, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

def sh(cmd, timeout=None, check=True):
    print("$", cmd)
    try: r = subprocess.run(cmd, shell=True, timeout=timeout).returncode
    except subprocess.TimeoutExpired: print("  …timed out"); r = -1
    if check: assert r == 0, f"command failed ({r}): {cmd}"
    return r

# The v3 disease model (uploaded zip)
V3 = os.environ.get("FAST_V3_DIR") if FAST_TEST else None
if not V3:
    hits = glob.glob("/kaggle/input/**/crop_disease_int8.pte", recursive=True)
    for z in ([] if hits else glob.glob("/kaggle/input/**/*.zip", recursive=True)):     # zip uploaded without unpacking
        shutil.unpack_archive(z, f"{TMP}/v3"); hits = glob.glob(f"{TMP}/v3/**/crop_disease_int8.pte", recursive=True)
        if hits: break
    assert hits, "crop_disease_int8.pte not found — Add Input ▸ Upload your ps1_vision_outputs_v3.zip"
    V3 = os.path.dirname(hits[0])
print("v3 model folder:", V3)

## 1 · PlantVillage — same classes, same split as v2

In [ ]:
if FAST_TEST:
    DATA_DIR = os.environ["FAST_DATA_DIR"]
else:
    cands = [p for p in glob.glob("/kaggle/input/**/color", recursive=True) if os.path.isdir(p)]
    if not cands:
        import kagglehub
        root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
        cands = [p for p in glob.glob(os.path.join(root, "**", "color"), recursive=True) if os.path.isdir(p)]
    assert cands, "PlantVillage 'color' folder not found — add the dataset via Add Input"
    DATA_DIR = cands[0]
print("DATA_DIR:", DATA_DIR)

classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
cls_idx = {c: i for i, c in enumerate(classes)}
def parse_label(raw):
    crop, _, cond = raw.partition("___")
    clean = lambda s: s.replace("_", " ").replace(",", "").strip()
    return {"raw": raw, "crop": clean(crop), "condition": clean(cond) or "unknown", "healthy": cond.lower() == "healthy"}
labels = [parse_label(c) for c in classes]
json.dump(labels, open(f"{OUT_DIR}/labels.json", "w"), indent=1)
crop_of = [l["crop"] for l in labels]
print(len(classes), "classes")
v3_labels = [l["raw"] for l in json.load(open(f"{V3}/labels.json"))]
assert v3_labels == classes, "v3 labels.json does not match the PlantVillage class order"

from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset, Dataset
pv_all = ImageFolder(DATA_DIR)
assert pv_all.classes == classes
idx = list(range(len(pv_all))); random.Random(SEED).shuffle(idx)          # identical to v2
n_val = n_test = int(0.1 * len(idx))
val_idx, test_idx, train_idx = idx[:n_val], idx[n_val:n_val + n_test], idx[n_val + n_test:]
json.dump({"data_dir": DATA_DIR, "classes": classes, "train": train_idx, "val": val_idx, "test": test_idx},
          open(f"{OUT_DIR}/splits.json", "w"))
print(f"PlantVillage: train {len(train_idx)} · val {len(val_idx)} · test {len(test_idx)}")

## 2 · PlantDoc — real field photos of the same crops
2,598 photos from the web/fields, 27 classes; 25 of them match a PlantVillage class. Labels are a bit noisy (it is a real-world dataset). We train on its **train** folder and report accuracy on its **test** folder only.

In [ ]:
PD2PV = {
 "apple scab leaf": "Apple___Apple_scab", "apple leaf": "Apple___healthy", "apple rust leaf": "Apple___Cedar_apple_rust",
 "bell pepper leaf": "Pepper,_bell___healthy", "bell pepper leaf spot": "Pepper,_bell___Bacterial_spot",
 "blueberry leaf": "Blueberry___healthy", "cherry leaf": "Cherry_(including_sour)___healthy",
 "corn gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
 "corn rust leaf": "Corn_(maize)___Common_rust_", "peach leaf": "Peach___healthy",
 "potato leaf early blight": "Potato___Early_blight", "potato leaf late blight": "Potato___Late_blight",
 "raspberry leaf": "Raspberry___healthy", "soyabean leaf": "Soybean___healthy", "squash powdery mildew leaf": "Squash___Powdery_mildew",
 "strawberry leaf": "Strawberry___healthy", "tomato early blight leaf": "Tomato___Early_blight",
 "tomato septoria leaf spot": "Tomato___Septoria_leaf_spot", "tomato leaf": "Tomato___healthy",
 "tomato leaf bacterial spot": "Tomato___Bacterial_spot", "tomato leaf late blight": "Tomato___Late_blight",
 "tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus", "tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
 "tomato mold leaf": "Tomato___Leaf_Mold", "grape leaf": "Grape___healthy", "grape leaf black rot": "Grape___Black_rot",
}
norm = lambda s: re.sub(r"[\s_]+", " ", s).strip().lower()

if FAST_TEST:
    PD_DIR = os.environ.get("FAST_PD_DIR", "")
else:
    PD_DIR = f"{TMP}/PlantDoc-Dataset"
    if not os.path.isdir(f"{PD_DIR}/test"):
        sh(f"git clone --depth 1 -q https://github.com/pratikkayal/PlantDoc-Dataset {PD_DIR}", timeout=900, check=False)

def ok_image(p):
    try:
        with Image.open(p) as im: im.draft("RGB", (256, 256)); im.convert("RGB").load()
        return True
    except Exception: return False

def pd_items(split):
    out, skipped = [], set()
    root = os.path.join(PD_DIR, split)
    if not os.path.isdir(root): return out
    for d in sorted(os.listdir(root)):
        tgt = PD2PV.get(norm(d))
        if tgt is None or tgt not in cls_idx: skipped.add(d); continue
        for f in sorted(os.listdir(os.path.join(root, d))):
            p = os.path.join(root, d, f)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")) and ok_image(p): out.append((p, cls_idx[tgt]))
    if skipped: print(f"  {split}: no PlantVillage match, skipped →", sorted(skipped))
    return out

pd_train_all, pd_test = pd_items("train"), pd_items("test")
HAVE_PD = len(pd_train_all) > 0 and len(pd_test) > 0
random.Random(SEED).shuffle(pd_train_all)
n_pdv = max(1, len(pd_train_all) // 10) if HAVE_PD else 0
pd_val, pd_train = pd_train_all[:n_pdv], pd_train_all[n_pdv:]
print(f"PlantDoc: train {len(pd_train)} · val {len(pd_val)} · test {len(pd_test)}" if HAVE_PD
      else "⚠️ PlantDoc not available — training continues with background swapping only (no real-photo numbers)")

## 4 · Leaf cut-out + background swapping (same code as notebook 01 v3)

In [ ]:
SEV = dict(BG_DIST=18, LESION_DIST=22, BORDER=6, KEEP_PIECE=0.15, MILD=0.10, MODERATE=0.25)
GREEN = dict(H_MIN=30, H_MAX=90, S_MIN=35)     # OpenCV hue 0–180 — same "greenish" test as the app

def _lab(im):
    return cv2.cvtColor(im, cv2.COLOR_RGB2LAB).astype(np.float32) * np.array([100/255, 1, 1], np.float32) - np.array([0, 128, 128], np.float32)

def _fill_holes(m):
    h, w = m.shape; pad = np.zeros((h + 2, w + 2), np.uint8); pad[1:-1, 1:-1] = m
    ff = pad.copy(); cv2.floodFill(ff, np.zeros((h + 4, w + 4), np.uint8), (0, 0), 1)
    holes = (ff == 0)[1:-1, 1:-1]
    return m.astype(bool) | holes

def leaf_mask(rgb, cfg=SEV):
    L = _lab(rgb); h, w = L.shape[:2]; b = cfg["BORDER"]
    ring = np.zeros((h, w), bool); ring[:b] = ring[-b:] = True; ring[:, :b] = ring[:, -b:] = True
    bg = np.median(L[ring], 0); d_bg = np.sqrt(((L - bg) ** 2).sum(-1))
    fg = (d_bg > cfg["BG_DIST"]).astype(np.uint8)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(fg, 8)
    if n <= 1: return np.zeros((h, w), bool), d_bg, L
    big = st[1:, cv2.CC_STAT_AREA].max()
    keep = np.isin(lab, [i for i in range(1, n) if st[i, cv2.CC_STAT_AREA] >= cfg["KEEP_PIECE"] * big])
    leaf = _fill_holes(keep.astype(np.uint8))
    if leaf.mean() > 0.97: leaf = np.ones((h, w), bool)
    return leaf, d_bg, L

def severity(rgb224, cfg=SEV):
    leaf, d_bg, L = leaf_mask(rgb224, cfg); h, w = leaf.shape
    if not leaf.any(): return 0.0, "No leaf", leaf, leaf
    core = cv2.erode(leaf.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
    hsv = cv2.cvtColor(rgb224, cv2.COLOR_RGB2HSV); H, S, V = [hsv[..., i].astype(int) for i in range(3)]
    greenish = core & (H >= GREEN["H_MIN"]) & (H <= GREEN["H_MAX"]) & (S >= GREEN["S_MIN"])
    ref = np.median(L[greenish], 0) if greenish.sum() > 50 else np.median(L[core], 0)
    shadow = ((V < 110) & (S < 55)) | (V < 45)
    les = core & (np.sqrt(((L - ref) ** 2).sum(-1)) > cfg["LESION_DIST"]) & (d_bg >= cfg["BG_DIST"]) & ~shadow
    les = cv2.morphologyEx(les.astype(np.uint8), cv2.MORPH_OPEN, np.ones((2, 2), np.uint8)).astype(bool)
    frac = float(les.sum()) / max(int(core.sum()), 1)
    return frac, ("Mild" if frac < cfg["MILD"] else "Moderate" if frac < cfg["MODERATE"] else "Severe"), les, leaf

def green_fraction(rgb224):
    """Share of the found 'leaf' that is plant-green. Real leaves: high. Faces, walls, clothes: near 0."""
    leaf, _, _ = leaf_mask(rgb224)
    if leaf.sum() < 0.05 * leaf.size: return 0.0
    hsv = cv2.cvtColor(rgb224, cv2.COLOR_RGB2HSV)
    g = (hsv[..., 0] >= GREEN["H_MIN"]) & (hsv[..., 0] <= GREEN["H_MAX"]) & (hsv[..., 1] >= GREEN["S_MIN"])
    return float((g & leaf).sum()) / float(leaf.sum())

def app_square(path_or_img, size=224):
    """Exactly what the app does: centre square → 512 → 224."""
    im = path_or_img if isinstance(path_or_img, np.ndarray) else np.array(_open(path_or_img))
    s = min(im.shape[:2]); y0 = (im.shape[0] - s) // 2; x0 = (im.shape[1] - s) // 2
    return cv2.resize(cv2.resize(im[y0:y0 + s, x0:x0 + s], (512, 512), interpolation=cv2.INTER_AREA), (size, size), interpolation=cv2.INTER_AREA)

def _open(p, draft=512):
    im = Image.open(p); im.draft("RGB", (draft, draft)); return im.convert("RGB")

PALETTE = np.array([[120,85,60],[90,65,45],[150,120,90],[70,55,40],      # soils
                    [60,110,50],[40,80,35],[100,140,70],                  # grass / foliage
                    [128,128,128],[200,200,200],[240,240,235],[60,60,60], # greys, paper, dark
                    [225,180,150],[160,110,80],                           # hands / skin
                    [150,190,230],[170,130,90],[200,170,120]], np.float32) # sky, wood, sand

def random_background(rng, size=256):
    r = rng.random()
    if r < 0.35 and pd_train:            # blurred real field photo
        im = np.array(_open(pd_train[rng.integers(len(pd_train))][0]))
        im = cv2.resize(im, (size, size)); k = int(rng.choice([7, 11, 15, 21]))
        return cv2.GaussianBlur(im, (k, k), 0)
    if r < 0.55 and TEXTURES:            # wall / fabric / wood texture
        return cv2.resize(np.array(_open(TEXTURES[rng.integers(len(TEXTURES))])), (size, size))
    if r < 0.70:                          # another lab leaf, heavily blurred (leafy clutter)
        im = np.array(_open(pv_all.samples[train_idx[rng.integers(len(train_idx))]][0]))
        return cv2.GaussianBlur(cv2.resize(im, (size, size)), (31, 31), 0)
    c1, c2 = PALETTE[rng.integers(len(PALETTE))], PALETTE[rng.integers(len(PALETTE))]
    t = np.linspace(0, 1, size)[:, None, None] if rng.random() < 0.5 else np.zeros((size, 1, 1))
    bg = c1 * (1 - t) + c2 * t + np.zeros((size, size, 3))
    blotch = cv2.resize(rng.normal(0, 1, (8, 8)).astype(np.float32), (size, size), interpolation=cv2.INTER_CUBIC)[..., None]
    grain = rng.normal(0, 1, (size, size, 1))          # same on all channels → natural texture, not rainbow speckle
    bg = bg * (1 + blotch * rng.uniform(0.03, 0.15)) + grain * rng.uniform(1, 8)
    return np.clip(bg, 0, 255).astype(np.uint8)

def swap_background(img, rng, size=256):
    a = cv2.resize(np.array(img), (size, size))
    leaf, _, _ = leaf_mask(a)
    if leaf.mean() < 0.05 or leaf.mean() > 0.95: return img
    m = cv2.GaussianBlur(leaf.astype(np.float32), (5, 5), 0)
    s = rng.uniform(0.55, 1.0); n = int(size * s)
    ang = rng.uniform(-180, 180); M = cv2.getRotationMatrix2D((size / 2, size / 2), ang, 1.0)
    a = cv2.resize(cv2.warpAffine(a, M, (size, size), borderValue=0), (n, n))
    m = cv2.resize(cv2.warpAffine(m, M, (size, size), borderValue=0), (n, n))[..., None]
    bg = random_background(rng, size).astype(np.float32)
    y, x = rng.integers(0, size - n + 1), rng.integers(0, size - n + 1)
    bg[y:y + n, x:x + n] = bg[y:y + n, x:x + n] * (1 - m) + a * m
    if rng.random() < 0.3:                # cast a shadow across part of the photo
        yy, xx = np.mgrid[0:size, 0:size]; th = rng.uniform(0, np.pi)
        side = (np.cos(th) * (xx - size / 2) + np.sin(th) * (yy - size / 2)) > rng.uniform(-60, 60)
        bg[side] *= rng.uniform(0.45, 0.8)
    return Image.fromarray(np.clip(bg, 0, 255).astype(np.uint8))

class RandomJPEG:
    def __init__(self, p=0.4): self.p = p
    def __call__(self, im):
        if random.random() > self.p: return im
        b = io.BytesIO(); im.save(b, "JPEG", quality=random.randint(25, 85)); b.seek(0); return Image.open(b).convert("RGB")

photo_tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.45, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(0.2),
                      T.ColorJitter(0.4, 0.4, 0.35, 0.04), T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.3),
                      RandomJPEG(0.4), T.ToTensor()])

## 3 · "Not a leaf" photos — downloaded once, resized to 256 px
85 % train the gate, 15 % are kept aside to measure how well it rejects them.

In [ ]:
NEG_DIR = f"{TMP}/neg"; os.makedirs(NEG_DIR, exist_ok=True)
def save_neg(arr_or_img, name):
    im = arr_or_img if isinstance(arr_or_img, Image.Image) else Image.fromarray(arr_or_img)
    im = im.convert("RGB"); w, h = im.size; s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((256, 256), Image.BILINEAR)
    p = f"{NEG_DIR}/{name}.jpg"; im.save(p, quality=90); return p

neg = {"everyday": [], "faces": [], "textures": []}
if FAST_TEST:
    rs = np.random.RandomState(0)
    for i in range(120):     # fake "everyday" photos: noisy colour blocks
        a = rs.randint(0, 255, (8, 8, 3)).astype(np.uint8); neg["everyday"].append(save_neg(cv2.resize(a, (256, 256)), f"ev{i}"))
    for i in range(40):
        a = np.full((256, 256, 3), (220, 170, 140), np.uint8); neg["faces"].append(save_neg(a, f"fa{i}"))
else:
    # COCO val2017: 5,000 everyday photos (people, crowds, sports fields, streets, rooms, animals, food)
    if not os.path.isdir(f"{TMP}/val2017"):
        sh(f"wget -q -O {TMP}/val2017.zip http://images.cocodataset.org/zips/val2017.zip", timeout=1200, check=False)
        sh(f"cd {TMP} && unzip -q -o val2017.zip", timeout=600, check=False)
    for p in sorted(glob.glob(f"{TMP}/val2017/*.jpg")):
        try: neg["everyday"].append(save_neg(_open(p, 512), "coco_" + os.path.basename(p)[:-4]))
        except Exception: pass
    if not neg["everyday"]:        # fallback if COCO is unreachable: CIFAR-100 (small but varied)
        print("COCO not reachable — using CIFAR-100 instead")
        ds = torchvision.datasets.CIFAR100(f"{TMP}/cifar", train=False, download=True)
        for i in range(5000): neg["everyday"].append(save_neg(ds[i][0].resize((256, 256), Image.BICUBIC), f"cif{i}"))
    try:
        from sklearn.datasets import fetch_lfw_people
        lfw = fetch_lfw_people(color=True, resize=1.0, min_faces_per_person=0, download_if_missing=True, data_home=f"{TMP}/lfw")
        for k, i in enumerate(np.random.RandomState(0).choice(len(lfw.images), min(3000, len(lfw.images)), replace=False)):
            im = lfw.images[i]; im = im if im.max() > 1.5 else im * 255
            neg["faces"].append(save_neg(im.astype(np.uint8), f"lfw{k}"))
    except Exception as e: print("faces skipped:", str(e)[:120])
    for split in ("train", "test"):
        sh(f"{sys.executable} -c \"import torchvision; torchvision.datasets.DTD('{TMP}/dtd', split='{split}', download=True)\"", timeout=600, check=False)
    for k, p in enumerate(sorted(set(glob.glob(f"{TMP}/dtd/**/images/**/*.jpg", recursive=True)))):
        try: neg["textures"].append(save_neg(Image.open(p), f"dtd{k}"))
        except Exception: pass
TEXTURES = []           # (used by random_background below; textures are negatives here, not leaf backgrounds)
split_neg = {}
for k, v in neg.items():
    v = sorted(v); random.Random(SEED).shuffle(v); n = max(1, int(0.15 * len(v))) if v else 0
    split_neg[k] = (v[n:], v[:n])
    print(f"{k:9s}: train {len(v) - n:5d} · held out {n:4d}")

## 5 · Gate training data
* **Yes (1):** PlantVillage training leaves (half on new backgrounds) + PlantDoc training photos.
* **No (0):** everyday photos, faces, textures, plain empty backgrounds — and 35 % of them made **green** (green tint or green paint blobs), so green colour alone never means "leaf".

In [ ]:
def plain_background(rng, size=256):
    c1, c2 = PALETTE[rng.integers(len(PALETTE))], PALETTE[rng.integers(len(PALETTE))]
    t = np.linspace(0, 1, size)[:, None, None] if rng.random() < 0.5 else np.zeros((size, 1, 1))
    bg = c1 * (1 - t) + c2 * t + np.zeros((size, size, 3))
    blotch = cv2.resize(rng.normal(0, 1, (8, 8)).astype(np.float32), (size, size), interpolation=cv2.INTER_CUBIC)[..., None]
    bg = bg * (1 + blotch * rng.uniform(0.03, 0.15)) + rng.normal(0, 1, (size, size, 1)) * rng.uniform(1, 8)
    return np.clip(bg, 0, 255).astype(np.uint8)

def make_green(a, rng):
    """Green tint (turf, green walls/clothes) or green paint blobs (Holi faces)."""
    a = a.copy()
    if rng.random() < 0.5:
        hsv = cv2.cvtColor(a, cv2.COLOR_RGB2HSV).astype(np.int32)
        mask = np.ones(a.shape[:2], bool) if rng.random() < 0.5 else (hsv[..., 1] > 30)   # whole photo, or its coloured parts
        hsv[..., 0] = np.where(mask, rng.integers(35, 80), hsv[..., 0])
        hsv[..., 1] = np.where(mask, np.clip(hsv[..., 1] + rng.integers(40, 120), 0, 255), hsv[..., 1])
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    over = a.astype(np.float32)
    for _ in range(rng.integers(2, 7)):
        m = np.zeros(a.shape[:2], np.uint8)
        cv2.ellipse(m, (int(rng.integers(0, 256)), int(rng.integers(0, 256))), (int(rng.integers(15, 90)), int(rng.integers(15, 90))),
                    float(rng.uniform(0, 180)), 0, 360, 1, -1)
        m = cv2.GaussianBlur(m.astype(np.float32), (21, 21), 0)[..., None] * rng.uniform(0.6, 0.95)
        colour = np.array([rng.integers(20, 90), rng.integers(150, 230), rng.integers(20, 90)], np.float32)
        over = over * (1 - m) + colour * m
    return np.clip(over, 0, 255).astype(np.uint8)

GREEN_PROB = 0.35
rs = random.Random(SEED)
pos_items = [(pv_all.samples[i][0], "pv") for i in rs.sample(train_idx, min(N_PV_POS, len(train_idx)))] + \
            [(p, "pd") for p, _ in pd_train] * PLANTDOC_REPEAT
neg_items = [(p, "neg") for k in split_neg for p in split_neg[k][0]] + [(None, "plain")] * (2000 if not FAST_TEST else 50)

class GateSet(Dataset):
    def __init__(self, pos, negs): self.items = [(p, s, 1) for p, s in pos] + [(p, s, 0) for p, s in negs]
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        p, src, y = self.items[k]; rng = np.random.default_rng(random.getrandbits(32))
        if src == "plain": im = Image.fromarray(plain_background(rng))
        else:
            im = _open(p).resize((256, 256))
            if src == "pv" and rng.random() < BG_SWAP_PROB: im = swap_background(im, rng)
            if y == 0 and rng.random() < GREEN_PROB: im = Image.fromarray(make_green(np.array(im), rng))
        return photo_tf(im), y

gate_train = GateSet(pos_items, neg_items)
workers = 0 if FAST_TEST else 4
gate_dl = DataLoader(gate_train, BATCH, shuffle=True, num_workers=workers, drop_last=True, pin_memory=True, persistent_workers=workers > 0)
print(f"gate training: {len(pos_items)} leaf photos · {len(neg_items)} not-leaf photos per epoch")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(3, 6, figsize=(15, 8))
for a in ax.flat:
    x, y = gate_train[random.randrange(len(gate_train))]
    a.imshow(x.permute(1, 2, 0).numpy()); a.set_title("LEAF ✓" if y else "not a leaf ✗", fontsize=9, color="green" if y else "red"); a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/gate_training_examples.png", dpi=100); plt.close()

## 6 · Train the gate (MobileNetV3-Small)

In [ ]:
GATE_SRC = """
import torch, torch.nn as nn, torchvision
class LeafGate(nn.Module):
    # 0-1 RGB in -> [p(not a leaf photo), p(crop leaf photo)] out (softmax in export mode)
    def __init__(self, pretrained=True):
        super().__init__()
        w = torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
        self.net = torchvision.models.mobilenet_v3_small(weights=w)
        self.net.classifier[3] = nn.Linear(self.net.classifier[3].in_features, 2)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.export_mode = False
    def forward(self, x):
        logits = self.net((x - self.mean) / self.std)
        return torch.softmax(logits, dim=1) if self.export_mode else logits
"""
open(f"{WORK}/gate_model.py", "w").write(GATE_SRC)
sys.path.insert(0, WORK)
from gate_model import LeafGate
import torch.nn as nn
gate = LeafGate(pretrained=not FAST_TEST).to(device)
opt = torch.optim.AdamW(gate.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(gate_dl))
use_amp = device == "cuda"; scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
crit = nn.CrossEntropyLoss(label_smoothing=0.05)
for ep in range(EPOCHS):
    gate.train(); t0 = time.time(); tot = n = 0
    for x, y in gate_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=use_amp): loss = crit(gate(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * x.size(0); n += x.size(0)
    print(f"epoch {ep+1}/{EPOCHS} · loss {tot/n:.4f} · {time.time()-t0:.0f}s")
torch.save(gate.state_dict(), f"{OUT_DIR}/gate_best.pt")

## 7 · Gate test — leaves it has never seen vs. "not a leaf" photos it has never seen
The cut-off is set so that **99 % of lab leaves and 98 % of real field leaves pass**; then we report how many non-leaf photos are rejected (normal and green-tinted).

In [ ]:
def as_app_input(arr_or_path):
    a = app_square(arr_or_path) if not isinstance(arr_or_path, np.ndarray) else cv2.resize(arr_or_path, (224, 224), interpolation=cv2.INTER_AREA)
    return torch.from_numpy(a).permute(2, 0, 1).float() / 255

@torch.no_grad()
def leaf_prob(inputs):          # paths or arrays, converted batch by batch to keep memory low
    gate.eval(); out = []
    for i in range(0, len(inputs), 128):
        x = torch.stack([as_app_input(v) for v in inputs[i:i + 128]]).to(device)
        out.append(torch.softmax(gate(x).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy() if out else np.array([])

rs = np.random.RandomState(1)
pv_test_s = [pv_all.samples[i][0] for i in rs.choice(test_idx, min(2000, len(test_idx)), replace=False)]
P_pv = leaf_prob(pv_test_s)
P_pd = leaf_prob([p for p, _ in pd_test]) if HAVE_PD else np.array([])
rng_eval = np.random.default_rng(7)
held = {k: [np.array(_open(p).resize((256, 256))) for p in split_neg[k][1]] for k in split_neg}
held["plain backgrounds"] = [plain_background(rng_eval) for _ in range(300 if not FAST_TEST else 20)]
held_green = {f"{k} (made green)": [make_green(a, rng_eval) for a in v] for k, v in held.items() if k != "plain backgrounds"}
P_neg = {k: leaf_prob(v) for k, v in {**held, **held_green}.items() if len(v)}

thr = float(np.percentile(P_pv, 1))
if len(P_pd): thr = min(thr, float(np.percentile(P_pd, 2)))
thr = float(np.clip(np.floor(thr * 100) / 100, 0.05, 0.5))
gate_report = {"threshold_min_leaf_prob": thr,
               "leaves_pass": {"lab (PlantVillage test)": float((P_pv >= thr).mean()),
                               "real field (PlantDoc test)": float((P_pd >= thr).mean()) if len(P_pd) else None},
               "not_leaf_rejected": {k: float((v < thr).mean()) for k, v in P_neg.items()},
               "n": {"lab": len(P_pv), "field": len(P_pd), **{k: len(v) for k, v in P_neg.items()}}}
print(json.dumps(gate_report, indent=1))

## 8 · Rare-class correction for the v3 disease model
The app will divide each class probability by *(how common that class was in training)^τ* and rescale. τ = 0 means no change.
τ is chosen on **validation** photos (lab balanced accuracy + real-photo accuracy); the **test** numbers show the effect honestly.

In [ ]:
train_counts = np.bincount([pv_all.targets[i] for i in train_idx], minlength=len(classes)).astype(np.float64)
V3_PD_REPEAT = 4      # notebook 01 v3 showed each PlantDoc photo 4× per epoch
train_counts += V3_PD_REPEAT * np.bincount([y for _, y in pd_train], minlength=len(classes))   # v3 used PlantDoc ×4
prior = train_counts / train_counts.sum()
json.dump({"prior": prior.tolist()}, open(f"{OUT_DIR}/class_prior.json", "w"))
sets = {"pv_val": [(pv_all.samples[i][0], pv_all.targets[i]) for i in val_idx], "pv_test": [(pv_all.samples[i][0], pv_all.targets[i]) for i in test_idx],
        "pd_val": pd_val, "pd_test": pd_test}
if FAST_TEST: sets = {k: v[:40] for k, v in sets.items()}
json.dump({k: [[p, int(y)] for p, y in v] for k, v in sets.items()}, open(f"{OUT_DIR}/disease_eval_sets.json", "w"))
print({k: len(v) for k, v in sets.items()})

## 9 · Export the gate to ExecuTorch + run the v3 disease .pte on the evaluation photos

In [ ]:
PINS = "executorch==1.6.0 torch==2.15.0 torchvision==0.29.2 torchao==0.18.0"
if not os.path.exists(f"{ENV_DIR}/bin/python"):
    sh(f"{sys.executable} -m pip install -q virtualenv"); sh(f"{sys.executable} -m virtualenv -q {ENV_DIR}")
pip = f"{ENV_DIR}/bin/pip install -q --default-timeout=120 --retries 10"
if not FAST_TEST:
    if subprocess.run(f"{pip} --extra-index-url https://download.pytorch.org/whl/cpu {PINS}", shell=True).returncode != 0:
        sh(f"{pip} {PINS}")
sh(f"{ENV_DIR}/bin/python -c \"import torch, executorch; print('export env: torch', torch.__version__)\"")

EXPORT_SRC = r"""
import sys, os, json, time, argparse
import numpy as np, torch
from PIL import Image
from torch.export import export
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from executorch.runtime import Runtime
ap = argparse.ArgumentParser(); ap.add_argument("--work"); ap.add_argument("--out"); ap.add_argument("--v3"); a = ap.parse_args()
sys.path.insert(0, a.work)
from gate_model import LeafGate

def app_tensor(p):   # centre square -> 512 -> 224, exactly like the app
    im = Image.open(p); im.draft("RGB", (512, 512)); im = im.convert("RGB")
    w, h = im.size; s = min(w, h); im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    im = im.resize((512, 512), Image.BOX).resize((224, 224), Image.BOX)
    return torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1).float().div(255)[None]

m = LeafGate(pretrained=False); m.load_state_dict(torch.load(f"{a.out}/gate_best.pt", map_location="cpu")); m.export_mode = True; m.eval()
example = (torch.rand(1, 3, 224, 224),)
cal = json.load(open(f"{a.out}/gate_calib.json"))
prepared = prepare_pt2e(export(m, example).module(), XNNPACKQuantizer().set_global(get_symmetric_quantization_config(is_per_channel=True)))
with torch.no_grad():
    for p in cal: prepared(app_tensor(p))
prog = to_edge_transform_and_lower(export(convert_pt2e(prepared), example), partitioner=[XnnpackPartitioner()]).to_executorch()
open(f"{a.out}/leaf_gate_int8.pte", "wb").write(prog.buffer); print(f"gate INT8: {len(prog.buffer)/1e6:.2f} MB", flush=True)

def run(path, items, out_name):
    meth = Runtime.get().load_program(path).load_method("forward"); P, ms = [], []
    for p in items:
        x = app_tensor(p); t = time.perf_counter(); P.append(meth.execute([x])[0].numpy()[0]); ms.append(time.perf_counter() - t)
    np.save(f"{a.out}/{out_name}.npy", np.array(P)); return 1000 * float(np.median(ms)) if ms else None

g = json.load(open(f"{a.out}/gate_eval_paths.json"))
gate_ms = run(f"{a.out}/leaf_gate_int8.pte", g["leaves"], "gate_int8_leaves")
run(f"{a.out}/leaf_gate_int8.pte", g["not_leaves"], "gate_int8_not_leaves")
ev = json.load(open(f"{a.out}/disease_eval_sets.json"))
for k, items in ev.items():
    run(f"{a.v3}/crop_disease_int8.pte", [p for p, _ in items], f"disease_{k}")
    print("disease probs saved:", k, len(items), flush=True)
json.dump({"gate_int8_median_ms_notebook_cpu": gate_ms, "gate_size_mb": os.path.getsize(f"{a.out}/leaf_gate_int8.pte") / 1e6},
          open(f"{a.out}/gate_pte_metrics.json", "w"))
"""
open(f"{WORK}/export_gate.py", "w").write(EXPORT_SRC)
json.dump([p for p, _ in pos_items[:150:3]] + [p for k in split_neg for p in split_neg[k][0][:20]], open(f"{OUT_DIR}/gate_calib.json", "w"))
# INT8 check on files: held-out leaves vs held-out not-leaf photos (normal colours)
json.dump({"leaves": pv_test_s[:600] + [p for p, _ in pd_test], "not_leaves": [p for k in split_neg for p in split_neg[k][1]]},
          open(f"{OUT_DIR}/gate_eval_paths.json", "w"))
sh(f"{ENV_DIR}/bin/python {WORK}/export_gate.py --work {WORK} --out {OUT_DIR} --v3 '{V3}'")

## 10 · Choose τ and the "Not sure" cut-off; check the INT8 gate

In [ ]:
ys = {k: np.array([y for _, y in v]) for k, v in sets.items()}
probs = {k: np.load(f"{OUT_DIR}/disease_{k}.npy") for k in sets}
def adjust(P, tau): Q = P / prior[None, :] ** tau; return Q / Q.sum(1, keepdims=True)
def bal_acc(P, y):
    pr = P.argmax(1); cl = np.unique(y); return float(np.mean([(pr[y == c] == c).mean() for c in cl])) if len(y) else float("nan")
acc_ = lambda P, y: float((P.argmax(1) == y).mean()) if len(y) else float("nan")
rows = []
for tau in [0.0, 0.25, 0.5, 0.75, 1.0]:
    r = {"tau": tau}
    for k in sets: Q = adjust(probs[k], tau); r[f"{k}_acc"] = acc_(Q, ys[k]); r[f"{k}_bal"] = bal_acc(Q, ys[k])
    rows.append(r)
base = rows[0]
def score(r): return 0.5 * r["pv_val_bal"] + 0.5 * (r["pd_val_acc"] if len(ys["pd_val"]) else r["pv_val_bal"])
ok = [r for r in rows if r["pv_val_acc"] >= base["pv_val_acc"] - 0.005]
best = max(ok, key=score)
TAU = best["tau"]
print("τ   lab-val acc / balanced · real-val acc ||  lab-test acc / balanced · real-test acc")
for r in rows:
    print(f"{r['tau']:.2f}  {r['pv_val_acc']:.4f} / {r['pv_val_bal']:.4f} · {r['pd_val_acc']:.3f}  ||  "
          f"{r['pv_test_acc']:.4f} / {r['pv_test_bal']:.4f} · {r['pd_test_acc']:.3f}" + ("   ← chosen" if r is best else ""))

# per-class recall on the lab test set, before → after (rare classes first)
Qb, Qa = probs["pv_test"], adjust(probs["pv_test"], TAU); yt = ys["pv_test"]
per_class = {classes[c]: {"n_test": int((yt == c).sum()), "recall_before": float((Qb[yt == c].argmax(1) == c).mean()),
                          "recall_after": float((Qa[yt == c].argmax(1) == c).mean())} for c in np.unique(yt)}
for c in sorted(per_class, key=lambda c: per_class[c]["n_test"])[:8]:
    v = per_class[c]; print(f"  {c:45s} n={v['n_test']:4d}  {v['recall_before']:.3f} → {v['recall_after']:.3f}")

# "Not sure" cut-off on the corrected probabilities: ~7.5 % of correct real-photo answers fall below it
Qv = adjust(probs["pd_val"] if len(ys["pd_val"]) else probs["pv_val"], TAU); yv = ys["pd_val"] if len(ys["pd_val"]) else ys["pv_val"]
correct = Qv.max(1)[Qv.argmax(1) == yv]
MIN_CONF = float(np.clip(round(np.percentile(correct, 7.5) / 0.05) * 0.05, 0.35, 0.6)) if len(correct) else 0.5
print("min_confidence:", MIN_CONF)

# INT8 gate check (from the .pte, normal-colour photos)
gl = np.load(f"{OUT_DIR}/gate_int8_leaves.npy")[:, 1]; gn = np.load(f"{OUT_DIR}/gate_int8_not_leaves.npy")[:, 1]
gate_report["int8_check"] = {"leaves_pass": float((gl >= thr).mean()), "not_leaves_rejected": float((gn < thr).mean()), "n": [len(gl), len(gn)]}
gate_report.update(json.load(open(f"{OUT_DIR}/gate_pte_metrics.json")))
print(json.dumps(gate_report["int8_check"]))

model_config = {"gate_min_leaf_prob": thr, "prior_tau": TAU, "class_prior": prior.tolist(), "min_confidence": MIN_CONF,
                "classes": classes}
json.dump(model_config, open(f"{OUT_DIR}/model_config.json", "w"), indent=1)
json.dump({"gate": gate_report, "prior_correction": {"rows": rows, "chosen_tau": TAU, "per_class_lab_test": per_class}},
          open(f"{OUT_DIR}/gate_metrics.json", "w"), indent=1, default=float)

## 11 · Package

In [ ]:
pkg = f"{WORK}/ps1_gate_outputs"; os.makedirs(pkg, exist_ok=True)
for f in ["leaf_gate_int8.pte", "model_config.json", "gate_metrics.json", "gate_training_examples.png"]: shutil.copy(f"{OUT_DIR}/{f}", pkg)
print("zipped →", shutil.make_archive(pkg, "zip", pkg))
print(f"\nGATE: {gate_report['gate_size_mb']:.2f} MB · cut-off {thr:.2f} · lab leaves pass {gate_report['leaves_pass']['lab (PlantVillage test)']:.3f}"
      f" · field leaves pass {gate_report['leaves_pass']['real field (PlantDoc test)']}")
print("not-a-leaf rejected:", {k: round(v, 3) for k, v in gate_report["not_leaf_rejected"].items()})
print(f"RARE-CLASS CORRECTION: τ = {TAU} · lab test balanced acc {base['pv_test_bal']:.4f} → {best['pv_test_bal']:.4f}"
      f" · real-photo test acc {base['pd_test_acc']:.3f} → {best['pd_test_acc']:.3f} · min_confidence {MIN_CONF}")

## 12 · FP32 gate (use this one in the app)
The INT8 gate lost accuracy (67 % of real leaves passed in the first run), so the app uses the FP32 gate (~6 MB, ~3 ms).

In [ ]:
# Why: the INT8 gate lost accuracy (only 67 % of real leaves passed). The gate is tiny, so FP32 is fine on the phone.
FP32_SRC = r"""
import sys, json, time, numpy as np, torch
from PIL import Image
from torch.export import export
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.runtime import Runtime
work, out = sys.argv[1], sys.argv[2]
sys.path.insert(0, work)
from gate_model import LeafGate
def app_tensor(p):
    im = Image.open(p); im.draft("RGB", (512, 512)); im = im.convert("RGB")
    w, h = im.size; s = min(w, h); im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    im = im.resize((512, 512), Image.BOX).resize((224, 224), Image.BOX)
    return torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1).float().div(255)[None]
m = LeafGate(pretrained=False); m.load_state_dict(torch.load(f"{out}/gate_best.pt", map_location="cpu")); m.export_mode = True; m.eval()
prog = to_edge_transform_and_lower(export(m, (torch.rand(1, 3, 224, 224),)), partitioner=[XnnpackPartitioner()]).to_executorch()
open(f"{out}/leaf_gate_fp32.pte", "wb").write(prog.buffer); print(f"gate FP32: {len(prog.buffer)/1e6:.2f} MB", flush=True)
meth = Runtime.get().load_program(f"{out}/leaf_gate_fp32.pte").load_method("forward")
g = json.load(open(f"{out}/gate_eval_paths.json")); res = {}
for k in ("leaves", "not_leaves"):
    ps, ms = [], []
    for p in g[k]:
        x = app_tensor(p); t = time.perf_counter(); ps.append(float(meth.execute([x])[0][0, 1])); ms.append(time.perf_counter() - t)
    res[k] = np.array(ps); print(k, len(ps), f"median {1000*np.median(ms):.1f} ms", flush=True)
rows = {}
for thr in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    rows[thr] = {"leaves_pass": float((res["leaves"] >= thr).mean()), "not_leaves_rejected": float((res["not_leaves"] < thr).mean())}
    print(f"cut-off {thr:.1f}: leaves pass {rows[thr]['leaves_pass']:.3f} · not-leaves rejected {rows[thr]['not_leaves_rejected']:.3f}")
json.dump({str(k): v for k, v in rows.items()}, open(f"{out}/gate_fp32_check.json", "w"), indent=1)
"""
open(f"{WORK}/gate_fp32.py", "w").write(FP32_SRC)
sh(f"{ENV_DIR}/bin/python {WORK}/gate_fp32.py {WORK} {OUT_DIR}")
pkg2 = f"{WORK}/ps1_gate_fp32"; os.makedirs(pkg2, exist_ok=True)
for f in ["leaf_gate_fp32.pte", "gate_fp32_check.json"]: shutil.copy(f"{OUT_DIR}/{f}", pkg2)
print("zipped →", shutil.make_archive(pkg2, "zip", pkg2))